
# Baseline Model: TF-IDF + Logistic Regression

This notebook builds a baseline for predicting the six Jigsaw toxicity labels. Comments can have multiple labels, so each label is treated as a separate binary classification task.

We reuse the saved 70/15/15 train, validation, and test partitions. TF-IDF is fitted only on training comments, and logistic regression uses balanced class weights to address label imbalance.

Validation performance is measured using average precision for each label, alongside its positive count. The test partition is reserved for final evaluation after model settings and decision thresholds are fixed.

In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import joblib 

df = pd.read_csv("../data/raw/train.csv")
test_ids = pd.read_csv("../data/processed/test_ids.csv")
train_ids = pd.read_csv("../data/processed/train_ids.csv")
validation_ids = pd.read_csv("../data/processed/validation_ids.csv")
run_id = 'baseline_v1_full_train'

labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

In [50]:
test_ids['partition'] = 'test'
train_ids['partition'] = 'train'
validation_ids['partition'] = 'validation'
id_maps = pd.concat([test_ids, train_ids, validation_ids])
id_maps

,id,partition
0,000113f07ec002fd,test
1,00040093b2687caa,test
2,0006f16e4e9f292e,test
3,00078f8ce7eb276d,test
4,0009801bd85e5806,test
...,...,...
23931,ff9ae4a64d2d5de1,validation
23932,ffb47123b2d82762,validation
23933,ffbdbb0483ed0841,validation
23934,ffc2f409658571f1,validation


In [ ]:
df = df.merge(
    on = 'id',
    how = 'left',
    validate = '1:1',
    right = id_maps 
)
df

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,partition
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0,train
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0,train
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0,test
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0,train
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0,train
...,...,...,...,...,...,...,...,...,...
159566,ffe987279560d7ff,""":::::And for the second time of asking, when ...",0,0,0,0,0,0,train
159567,ffea4adeee384e90,You should be ashamed of yourself \n\nThat is ...,0,0,0,0,0,0,train
159568,ffee36eab5c267c9,"Spitzer \n\nUmm, theres no actual article for ...",0,0,0,0,0,0,train
159569,fff125370e4aaaf3,And it looks like it was actually you who put ...,0,0,0,0,0,0,test


In [59]:
missing_partitions = df['partition'].isna()
blank_partitions = df['partition'].fillna('').str.strip().eq('')

print("Missing:", missing_partitions.sum())
print("Blank, excluding missing:", (blank_partitions & ~missing_partitions).sum())
print("Total Count: ", df['partition'].value_counts())

Missing: 0
Blank, excluding missing: 0
Total Count:  partition
train         111699
test           23936
validation     23936
Name: count, dtype: int64


In [78]:
label_sums = df.groupby('partition')[labels].sum()

label_sums = label_sums.loc[['train', 'validation', 'test']].T
label_sums

partition,train,validation,test
toxic,10706,2294,2294
severe_toxic,1116,240,239
obscene,5914,1267,1268
threat,335,71,72
insult,5514,1182,1181
identity_hate,983,211,211


Added a partition column mapped to the source dataset

Validated the results to make sure they match our expected values